# 01 — Orbital ground track

This browser-safe notebook recreates one core idea behind [Bilawal Sidhu's **God's Eye View**](https://github.com/bilawalsidhu/gods-eye-view): turning orbital state into a position on a rotating Earth. The upstream application uses **CelesTrak** two-line elements and `satellite.js`/SGP4 for operational satellite propagation. Here we intentionally use a simpler circular two-body model with synthetic elements so the example runs fully offline in JupyterLite.

**Upstream attribution:** God's Eye View © 2026 Bilawal Sidhu, MIT-licensed source code. Third-party data remains under provider terms; see the upstream [`DATA_SOURCES.md`](https://github.com/bilawalsidhu/gods-eye-view/blob/main/DATA_SOURCES.md).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

MU = 398600.4418      # km^3/s^2, Earth gravitational parameter
R_E = 6378.137        # km, equatorial radius
OMEGA_E = 7.2921159e-5 # rad/s, Earth rotation rate

altitude_km = 550.0
inclination_deg = 51.6
raan_deg = 20.0
phase_deg = 0.0

a = R_E + altitude_km
mean_motion = np.sqrt(MU / a**3)
period_s = 2*np.pi / mean_motion
print(f"Circular-orbit period: {period_s/60:.1f} minutes")


In [ ]:
def ground_track(times_s, altitude_km=550.0, inclination_deg=51.6, raan_deg=20.0, phase_deg=0.0):
    a = R_E + altitude_km
    n = np.sqrt(MU / a**3)
    i = np.deg2rad(inclination_deg)
    raan = np.deg2rad(raan_deg)
    u = n*times_s + np.deg2rad(phase_deg)

    # Circular orbit in an inertial frame after inclination + RAAN rotations.
    x = a*(np.cos(raan)*np.cos(u) - np.sin(raan)*np.sin(u)*np.cos(i))
    y = a*(np.sin(raan)*np.cos(u) + np.cos(raan)*np.sin(u)*np.cos(i))
    z = a*(np.sin(u)*np.sin(i))

    # Rotate inertial longitude into Earth-fixed longitude.
    theta = OMEGA_E * times_s
    x_e = np.cos(theta)*x + np.sin(theta)*y
    y_e = -np.sin(theta)*x + np.cos(theta)*y
    lon = np.rad2deg(np.arctan2(y_e, x_e))
    lat = np.rad2deg(np.arctan2(z, np.sqrt(x_e**2 + y_e**2)))
    return lat, lon

t = np.linspace(0, 3*period_s, 900)
lat, lon = ground_track(t, altitude_km, inclination_deg, raan_deg, phase_deg)

# Break the plotted line at the ±180° longitude discontinuity.
lon_plot = lon.copy()
lon_plot[1:][np.abs(np.diff(lon)) > 180] = np.nan

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(lon_plot, lat, linewidth=1.5)
ax.scatter(lon[0], lat[0], s=45, label="start")
ax.set(xlim=(-180,180), ylim=(-90,90), xlabel="Longitude (deg)", ylabel="Latitude (deg)", title="Synthetic low-Earth-orbit ground track — 3 orbits")
ax.set_xticks(np.arange(-180,181,60)); ax.set_yticks(np.arange(-90,91,30))
ax.grid(True, alpha=.3); ax.legend(); plt.show()


## What changes in the full upstream application?

God's Eye View ingests real satellite orbital elements, converts them to satellite records with `satellite.js`, propagates them with SGP4, and renders them on a Cesium globe. This notebook is therefore a **conceptual, original Python analogue**, not a port of the upstream JavaScript. Try changing altitude, inclination, RAAN, or phase to see how the track changes.
